# Preprocessing - Wine Quality

This notebook uses the original dataset located un `Dataset/winequality-white.csv`.

This program:

1. Loads csv.
2. Separates attributes and labels.
3. Splits dataset.
4. Scale the attributes.
5. Augmentates data.
6. Save datasets in respective folders on `Dataset`.

## Imports

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

## 1. Load dataset

In [5]:
# Only white wine was selected because this dataset has more instances
df = pd.read_csv("../Dataset/winequality-white.csv", sep=";")

## 2. Separate attributes and labels

In [ ]:
# Attributes
X = df.drop("quality", axis=1)

# Label
y = df["quality"]

print(X.head(2))
print(y.head(2))

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.0              0.27         0.36            20.7      0.045   
1            6.3              0.30         0.34             1.6      0.049   

   free sulfur dioxide  total sulfur dioxide  density   pH  sulphates  alcohol  
0                 45.0                 170.0    1.001  3.0       0.45      8.8  
1                 14.0                 132.0    0.994  3.3       0.49      9.5  
0    6
1    6
Name: quality, dtype: int64


## 3. Split the dataset

The split is 60% train, 20% validation & 20% test

In [7]:
# SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    # random seed
    random_state=42,
    # similar distribution between train and test
    stratify=y
)

## 4. Scale the attributes

In [8]:
# Scalability ONLY to train data
# Why? Besides from dataleakage, we don't want the model to ignore data because of different scales
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# Print how many instances of each class we have in the train set
print(y_train.value_counts())

quality
6    1758
5    1166
7     704
8     140
4     130
3      16
9       4
Name: count, dtype: int64


As SMOTE uses nearest neighbors to generate synthetic samples between instances of the same class, our smallest class (9) only contains 4 instances. Therefore, we need to reduce the default value of `k_neighbors` from 5 to 3.

In general:

k_neighbors = (smallest number of instances in a class) - 1

## 5. Data augmentation

In [13]:
# AUGMENTATION

smote = SMOTE(
    random_state=42,
    k_neighbors=3
)

X_train_aug, y_train_aug = smote.fit_resample(
    X_train_scaled,
    y_train
)
# Check size of the datasets
print("Original train:", X_train.shape)
print("Augmented train:", X_train_aug.shape)
print("Test:", X_test.shape)

Original train: (3918, 11)
Augmented train: (12306, 11)
Test: (980, 11)


## 6. Save tuned datasets (train & test)

In [12]:
# Save datasets in corresponding folders

# Convert augmented train to DataFrame
X_train_aug_df = pd.DataFrame(
    X_train_aug,
    columns=X_train.columns
)

y_train_aug_df = pd.DataFrame(
    y_train_aug,
    columns=["quality"]
)

train_aug_df = pd.concat([X_train_aug_df, y_train_aug_df], axis=1) # axis = 1: organize categories by columns

# Save augmented train
train_aug_df.to_csv("../Dataset/train/train.csv", index=False)

# Save test set
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1) # reset_index: avoid index problems when concatenating and drop=True: avoid adding the old index as a new column
test_df.to_csv("../Dataset/test/test.csv", index=False)
